# 🚀 AIC 2026: Vietnamese ASR Transcript Refinement Arena & Dual-GPU Pipeline

This notebook provides:
1. **🥊 Model Evaluation Arena**: Benchmarks the top 4 candidate models side-by-side on the exact same challenging Vietnamese segments, reporting **execution time**, **throughput (tokens/sec)**, and **refinement quality**.
2. **⚡ Dual-GPU Production Pipeline**: Runs the user-selected model in parallel across **2× Tesla T4 GPUs** (`cuda:0` and `cuda:1`) with automatic manifest merging and zip artifact export.

---
### 🏆 Evaluated Candidate Models:
* **`sail/Sailor2-8B-Chat`** *(Southeast Asian & Vietnamese Specialized SOTA)*
* **`Qwen/Qwen2.5-7B-Instruct`** *(Global Multilingual SOTA 7B)*
* **`SeaLLMs/SeaLLMs-v3-7B-Chat`** *(Southeast Asian Specialized 7B)*
* **`Qwen/Qwen3-14B` (4-bit NF4)** *(Frontier 14B under 4-bit quantization)*

In [ ]:
# Step 1: Verify Dual Tesla T4 GPU Accelerators
!nvidia-smi

In [ ]:
# Step 2: Install Required Packages (~10-15s, upgrades transformers for Qwen2/Qwen3 support)
!pip install -q --upgrade transformers accelerate bitsandbytes huggingface-hub

In [ ]:
# Step 3: Setup Codebase Repository
import os

if not os.path.exists("scripts"):
    !git clone https://github.com/TotallyNotMinh/aic2026.git
    %cd aic2026
else:
    !git pull

print("✓ Working directory:", os.getcwd())

## 🥊 Phase 1: Benchmark Arena (Evaluate Top 4 Models on Same Segments)

Runs all 4 models sequentially on GPU 0 with automatic VRAM garbage collection between runs. Measures **load time**, **generation time**, **tokens/sec**, and prints side-by-side Vietnamese outputs for your visual inspection.

In [ ]:
# Step 4: Run Candidate Model Evaluation Arena
!python scripts/evaluate_candidate_models.py --device cuda:0

## ⚡ Phase 2: Dual-GPU Full Dataset Refinement

Select your winning model from the arena above by setting `SELECTED_MODEL` and `SELECTED_QUANTIZATION`, then run the cells below to process all 294 transcript files across **2× Tesla T4 GPUs** in parallel.

In [ ]:
# Step 5: Choose Your Preferred Model
# Options:
#   1. "sail/Sailor2-8B-Chat"      | quantization: None
#   2. "Qwen/Qwen2.5-7B-Instruct"  | quantization: None
#   3. "SeaLLMs/SeaLLMs-v3-7B-Chat"| quantization: None
#   4. "Qwen/Qwen3-14B"            | quantization: "4bit"

SELECTED_MODEL = "sail/Sailor2-8B-Chat"     # Change if you prefer another candidate
SELECTED_QUANTIZATION = None              # Set to "4bit" if using Qwen3-14B, else None

print(f"Selected Model for Full Run: {SELECTED_MODEL} (Quantization: {SELECTED_QUANTIZATION})")

In [ ]:
# Step 6: Launch Dual-GPU Parallel Refinement Pipeline
import subprocess
import sys

cmd = [
    sys.executable, "scripts/run_dual_gpu_refinement.py",
    "--model-id", SELECTED_MODEL,
    "--transcripts-dir", "asr_transcripts/cache/asr_transcripts",
    "--output-dir", "asr_transcripts/cache/asr_transcripts",
    "--manifest-path", "cache/refinement_manifest.json",
    "--batch-size", "12",
    "--num-gpus", "2"
]

if SELECTED_QUANTIZATION:
    cmd.extend(["--quantization", SELECTED_QUANTIZATION])

subprocess.run(cmd, check=True)

In [ ]:
# Step 7: Display Unified Manifest Summary & Statistics
!python scripts/refine_transcripts_qwen.py --status --manifest-path "cache/refinement_manifest.json"

In [ ]:
# Step 8: Zip Output Transcripts for Download / Submission
!zip -q -r refined_asr_transcripts.zip asr_transcripts/cache/asr_transcripts/
!ls -lh refined_asr_transcripts.zip
print("✓ Output archive 'refined_asr_transcripts.zip' generated successfully!")